# Otso Vision v2 — Binarization Model Training

Notebook ini dikonfigurasi untuk **Google Colab (T4 GPU)**. Pastikan Runtime type diset ke GPU sebelum menjalankan.
Notebook ini akan mengambil dataset dari Home Server, melatih model binarization MobileNetV3-Small, lalu mengonversi ke `.tflite`.

## 1. Setup Environment

In [ ]:
!pip install -q tensorflow datasets pillow matplotlib

## 2. Sync Dataset dari Home Server
Gunakan `rclone` atau HTTP untuk mendownload dataset CORD/DIBCO yang sudah disiapkan di home server (100.90.222.22).

In [ ]:
# Contoh menggunakan SSH/SCP jika port terbuka, atau rclone:
# !sudo apt-get install sshpass -y
# !sshpass -p 'wisesa@karta' scp -o StrictHostKeyChecking=no -r wisesa@100.90.222.22:/home/wisesa/otso-vision/datasets/cord /content/datasets/

## 3. Data Pipeline (TF Dataset)
Muat gambar dan label.

In [ ]:
import tensorflow as tf
import os

IMG_SIZE = (256, 256)
BATCH_SIZE = 32

# (Fungsi loading tf.data.Dataset akan diimplementasikan di sini)

## 4. MobileNetV3-Small Binarization Model

In [ ]:
def build_model():
    base_model = tf.keras.applications.MobileNetV3Small(input_shape=(256, 256, 3), include_top=False, weights='imagenet')
    base_model.trainable = True
    
    inputs = tf.keras.Input(shape=(256, 256, 3))
    x = base_model(inputs)
    
    # Simple decoder for binary mask
    x = tf.keras.layers.Conv2DTranspose(128, 3, strides=2, padding='same', activation='relu')(x)
    x = tf.keras.layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x)
    x = tf.keras.layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)
    x = tf.keras.layers.Conv2DTranspose(16, 3, strides=2, padding='same', activation='relu')(x)
    x = tf.keras.layers.Conv2DTranspose(8, 3, strides=2, padding='same', activation='relu')(x)
    outputs = tf.keras.layers.Conv2D(1, 1, activation='sigmoid')(x) # Binary output
    
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = build_model()
model.summary()

## 5. TFLite Conversion & Quantization

In [ ]:
# converter = tf.lite.TFLiteConverter.from_keras_model(model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model = converter.convert()
# 
# with open('otso_vision_v2.tflite', 'wb') as f:
#     f.write(tflite_model)